# Model Evaluation with PetFinder Adoption Data

**Student Version**

**Shanshan (Shirley) Liu, Ph.D.**

**Summer Academy Afternoon Session, Part 1**  
**Time:** 2:00–3:30 PM  
**Main question:** *Can we predict whether a pet will be adopted within the first week?*

This notebook uses the PetFinder Adoption Prediction dataset as a classification example for model evaluation. The original Kaggle task predicts `AdoptionSpeed` with five categories. In this class, we simplify the task into a binary classification problem so that we can focus deeply on accuracy, confusion matrices, precision, recall, and F1 score.

---

## Part 1 plan

1. Load and inspect the PetFinder dataset  
2. Define a prediction target: `FastAdoption`  
3. Explore target and feature distributions  
4. Build a preprocessing pipeline  
5. Train a first classifier  
6. Evaluate with accuracy, confusion matrix, precision, recall, and F1 score  

---


## 1. Import modules

We import everything the notebook needs up front. Don't worry about what each tool does yet; we introduce each one where we first use it.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay,
    classification_report,
    precision_score,
    recall_score,
    f1_score,
)

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)

## 2. Load the PetFinder dataset

This notebook uses the PetFinder Adoption Prediction dataset from Kaggle:

https://www.kaggle.com/competitions/petfinder-adoption-prediction

We assume `petfinder.csv` is in the same directory as this notebook (the morning session created it for you).

In [ ]:
raw_df = pd.read_csv("petfinder.csv")
raw_df.head()

## 3. Inspect the dataset

## Dataset columns

The PetFinder dataset contains one row for each pet adoption profile. Some columns are numerical, while many columns are categorical variables represented by integer codes.

Full field definitions are on the Kaggle data page: [Kaggle PetFinder Adoption Prediction · Data](https://www.kaggle.com/competitions/petfinder-adoption-prediction/data).

### Pet identity and profile information

| Column | Meaning |
| --- | --- |
| `PetID` | Unique ID of the pet profile |
| `Name` | Name of the pet. Empty if the pet is not named |
| `Description` | Text description written for the pet profile. Most descriptions are in English, but some may be in Malay or Chinese |
| `RescuerID` | Unique ID of the rescuer or organization posting the profile |

### Pet characteristics

| Column | Meaning |
| --- | --- |
| `Type` | Type of animal: `1` = Dog, `2` = Cat |
| `Age` | Age of the pet when listed, **in months** |
| `Breed1` | Primary breed of the pet (refers to the breed-label dictionary) |
| `Breed2` | Secondary breed, if the pet is a mixed breed |
| `Gender` | `1` = Male, `2` = Female, `3` = Mixed (when the profile represents a group of pets) |
| `Color1` | Primary color of the pet |
| `Color2` | Secondary color of the pet |
| `Color3` | Third color of the pet |
| `MaturitySize` | Size at maturity: `0` = Not Specified, `1` = Small, `2` = Medium, `3` = Large, `4` = Extra Large |
| `FurLength` | `0` = Not Specified, `1` = Short, `2` = Medium, `3` = Long |
| `Health` | `0` = Not Specified, `1` = Healthy, `2` = Minor Injury, `3` = Serious Injury |

### Care and adoption information

| Column | Meaning |
| --- | --- |
| `Vaccinated` | `1` = Yes, `2` = No, `3` = Not Sure |
| `Dewormed` | `1` = Yes, `2` = No, `3` = Not Sure |
| `Sterilized` | Spayed / neutered: `1` = Yes, `2` = No, `3` = Not Sure |
| `Quantity` | Number of pets represented in the profile |
| `Fee` | Adoption fee in MYR (Malaysian Ringgit). `0` = free |
| `State` | State location in Malaysia (refers to the state-label dictionary) |
| `VideoAmt` | Number of videos uploaded for this profile |
| `PhotoAmt` | Number of photos uploaded for this profile |




> When looking at the first few rows, pay attention to two things:

1. Which columns are true numerical features, such as `Age`, `Fee`, `PhotoAmt`, and `VideoAmt`?
2. Which columns are categorical codes, such as `Type`, `Gender`, `Vaccinated`, and `State`?

This distinction matters because different types of features require different preprocessing steps.



In [ ]:
print("Dataset shape:", raw_df.shape)
print("\nColumn names:")
print(raw_df.columns.tolist())

raw_df.head()

## 4. Understand the original target

The original target is `AdoptionSpeed`. Smaller values mean faster adoption. For this class, we will simplify the task into a binary classification problem later.

### Target variable

- `AdoptionSpeed`: The adoption speed category. **Lower values mean faster adoption**. This is the original prediction target in the Kaggle competition.


In [ ]:
counts = raw_df["AdoptionSpeed"].value_counts().sort_index()
percentages = raw_df["AdoptionSpeed"].value_counts(normalize=True).sort_index() * 100

adoption_distribution = pd.DataFrame({
    "count": counts,
    "percentage": percentages.round(2)
})

adoption_distribution

## 5. Create a binary classification target

We define `FastAdoption = 1` if the pet was adopted on the same day or within the first week. Otherwise, `FastAdoption = 0`.

In [ ]:
df = raw_df.copy()

# FastAdoption = 1 means adopted on the same day or within the first week.
# FastAdoption = 0 means adopted later than the first week, or not adopted after 100 days.
df["FastAdoption"] = (df["AdoptionSpeed"] <= 1).astype(int)

df[["AdoptionSpeed", "FastAdoption"]].head(10)

In [ ]:
fast_counts = df["FastAdoption"].value_counts().sort_index()
fast_percentages = df["FastAdoption"].value_counts(normalize=True).sort_index() * 100

fast_distribution = pd.DataFrame({
    "count": fast_counts,
    "percentage": fast_percentages.round(2)
})

fast_distribution.index = ["Slow adoption", "Fast adoption"]
fast_distribution

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
df["FastAdoption"].value_counts().sort_index().plot(kind="bar", ax=ax)
ax.set_xticklabels(["Slow adoption", "Fast adoption"], rotation=0)
ax.set_ylabel("Number of pets")
ax.set_title("Binary target distribution")
plt.show()

## 6. Quick exploratory data analysis

Before training a machine learning model, we first explore the data to understand the prediction problem.

In this section, we ask a few simple questions:

1. Are dogs and cats adopted at different speeds?
2. Are younger pets adopted faster than older pets?
3. Do profiles with more photos tend to have faster adoption?
4. Does adoption fee appear to be related to adoption speed?

These exploratory checks help us decide which features may be useful for modeling.

One caution applies to everything in this section: these patterns are **associations, not causal effects**. If pets with more photos are adopted faster, that does not mean uploading more photos *causes* faster adoption. Other factors such as age, breed, health, location, or rescuer quality may drive both. The goal here is to build intuition about the dataset before fitting models, not to prove causation.

In [ ]:
# Helpful labels for a few encoded categorical variables.
type_map = {1: "Dog", 2: "Cat"}
gender_map = {1: "Male", 2: "Female", 3: "Mixed"}

df["TypeLabel"] = df["Type"].map(type_map)
df["GenderLabel"] = df["Gender"].map(gender_map)

### 6.1 Dogs vs. cats

First, we compare the proportion of fast adoption between dogs and cats.

In [ ]:
pd.crosstab(
    df["TypeLabel"],
    df["FastAdoption"],
    normalize="index"
).rename(columns={0: "Slow adoption", 1: "Fast adoption"}).round(3)

In this dataset, cats appear to have a higher fast-adoption rate than dogs. As noted above, this is an association only: age, breed, health, photos, and fee also differ between cats and dogs.

### 6.2 Gender and adoption speed

We can also compare adoption outcomes across gender categories.

In this dataset:

- `Male` means the profile is for a male pet.
- `Female` means the profile is for a female pet.
- `Mixed` means the profile represents a group of pets with mixed genders.

The table below shows the proportion of slow and fast adoption within each gender category.

In [ ]:
pd.crosstab(
    df["GenderLabel"],
    df["FastAdoption"],
    normalize="index"
).rename(columns={0: "Slow adoption", 1: "Fast adoption"}).round(3)

This table helps us check whether adoption outcomes differ across male, female, and mixed-gender profiles.

Remember that `Mixed` usually means the listing represents a group of pets, not a single pet. Differences for the `Mixed` category may therefore reflect group listings rather than gender itself.

In [ ]:
pd.crosstab(
    [df["TypeLabel"], df["GenderLabel"]],
    df["FastAdoption"],
    normalize="index"
).rename(columns={0: "Slow adoption", 1: "Fast adoption"}).round(3)

### 6.3 Numerical features by adoption outcome

Next, we compare several numerical features between slow-adopted and fast-adopted pets.

The table below shows the **average value** of each feature for the two adoption groups.

- `Age`: pet age in months
- `Fee`: adoption fee
- `PhotoAmt`: number of uploaded photos
- `VideoAmt`: number of uploaded videos

This helps us see whether fast-adopted pets look different from slow-adopted pets on average.

In [ ]:
summary_by_target = df.groupby("FastAdoption")[["Age", "Fee", "PhotoAmt", "VideoAmt"]].mean()
summary_by_target.index = ["Slow adoption", "Fast adoption"]
summary_by_target.round(2)

From this table, we can make a few observations:

- Fast-adopted pets are younger on average: about **8.72 months**, compared with **10.98 months** for slow-adopted pets.
- The average adoption fee is very similar between the two groups: about **21–22**.
- Slow-adopted pets have slightly more uploaded photos on average: **3.95** compared with **3.68**.
- Videos are rare in both groups, with average values close to zero.

These differences are relatively small, especially for fee, photos, and videos. Age shows a clearer difference, but we should still be careful: this table only compares simple averages and does not control for other features.

## 7. Select features for modeling

Now we choose which columns will be used as input features for the machine learning models.

In supervised learning, we usually separate the data into:

- `X`: the input features used to make predictions
- `y`: the target variable we want to predict

In this notebook, our target variable is:

- `FastAdoption`: whether a pet was adopted within the first week

We use a manageable set of tabular features so that we can focus on model evaluation and model selection, rather than text processing, image processing, or advanced feature engineering.

We divide the input features into two groups:

- **Numerical features**: `Age`, `Quantity`, `Fee`, `VideoAmt`, `PhotoAmt`
- **Categorical features**: `Type`, `Gender`, `MaturitySize`, `FurLength`, `Vaccinated`, `Dewormed`, `Sterilized`, `Health`, `Breed1`, `Color1`, `State`

Many categorical features are stored as numbers, but they are not continuous numerical values. For example, `Type = 1` means dog and `Type = 2` means cat. These values are category labels, not quantities. We will handle them as categorical features using one-hot encoding later.


In [ ]:
# We use tabular features only in this class.
# We intentionally exclude Description, Name, PetID, RescuerID, and image features.

numeric_features = [
    "Age", "Quantity", "Fee", "VideoAmt", "PhotoAmt"
]

categorical_features = [
    "Type", "Gender", "MaturitySize", "FurLength",
    "Vaccinated", "Dewormed", "Sterilized", "Health",
    "Breed1", "Color1", "State"
]

target = "FastAdoption"

model_df = df[numeric_features + categorical_features + [target]].copy()

X = model_df.drop(columns=[target])
y = model_df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

> **What do the shapes mean?**
>
> We have **14,993 pet profiles**, each described by **16 input features** (`X`) and one target value (`y`, `FastAdoption`).

> **Discussion: Why do we exclude some columns?**

For each column below, discuss whether it should be included as an input feature for this introductory model.

- `PetID`
- `RescuerID`
- `Name`
- `Description`
- image-related files
- `AdoptionSpeed`

Questions to consider:

1. Is this information available at prediction time?
2. Is it directly useful for a simple tabular model?
3. Would it require text processing, image processing, or additional feature engineering?
4. Could including it accidentally give the model the answer?

---


## 8. Train/test split

Before training a model, we split the dataset into two parts:

- **Training set**: used to train the model
- **Test set**: held out and used only for final evaluation

This is important because we want to evaluate how well the model performs on new, unseen data. If we evaluate the model on the same data used for training, the performance may look too optimistic.

In this notebook, we use 80% of the data for training and 20% for testing.

**Note:**
The test set will be held out until final evaluation. Model comparison and hyperparameter tuning should use the training set only.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y,   # what is this for?
)

print("Training set:", X_train.shape)
print("Test set:", X_test.shape)

In [ ]:
print("Training target distribution:")
print(y_train.value_counts(normalize=True).sort_index().round(3))

print("\nTest target distribution:")
print(y_test.value_counts(normalize=True).sort_index().round(3))


> Why do we check the target distribution?




Our target variable is not perfectly balanced:

- About **76.7%** of pets are slow adoption
- About **23.3%** of pets are fast adoption

Because of this, we use `stratify=y` when splitting the data.

This makes sure that the training set and test set have similar proportions of slow-adoption and fast-adoption pets.

In our result:

- Training set: 76.7% slow adoption, 23.3% fast adoption
- Test set: 76.7% slow adoption, 23.3% fast adoption

This is good because the test set still represents the original prediction problem.

<details>
<summary><b>Discussion question</b> (optional — click to expand)</summary>

What might go wrong if the test set accidentally contained very few fast-adoption pets?

</details>

## 9. Check for missing values before preprocessing

Before building the preprocessing pipeline, we should check whether the selected modeling features contain missing values.

This matters because many scikit-learn models, including Logistic Regression, KNN, MLPClassifier, and many tree-based models, do not accept `NaN` values directly. If missing values are present, the model may fail before it ever starts learning.

Even if this selected feature set has few or no missing values today, including imputation in the pipeline is still a good habit. The imputer is fit only on the training data inside the pipeline, which helps avoid data leakage from the test set.


In [ ]:
missing_summary = pd.DataFrame({
    "train_missing_count": X_train.isna().sum(),
    "train_missing_percent": (X_train.isna().mean() * 100).round(2),
    "test_missing_count": X_test.isna().sum(),
    "test_missing_percent": (X_test.isna().mean() * 100).round(2),
}).sort_values("train_missing_count", ascending=False)

print("Total missing values in selected training features:", int(X_train.isna().sum().sum()))
missing_summary


## 10. Build a preprocessing pipeline

Before training machine learning models, we need to convert the raw data into a format that the models can use.

In this dataset, we have two types of input features:

1. **Numerical features**, such as `Age`, `Fee`, `PhotoAmt`, and `VideoAmt`
2. **Categorical features**, such as `Type`, `Gender`, `Vaccinated`, `Health`, and `State`

These two types of features need different preprocessing steps.

### Numerical features

For numerical features, we use two steps:

- `SimpleImputer(strategy="median")`: fills missing numerical values using the median value of that column.
- `StandardScaler()`: standardizes numerical features so they have a similar scale.

Scaling is useful for models such as Logistic Regression, KNN, and MLPClassifier because these models can be sensitive to feature scale.

### Categorical features

For categorical features, we also use two steps:

- `SimpleImputer(strategy="most_frequent")`: fills missing categorical values using the most common category.
- `OneHotEncoder(handle_unknown="ignore")`: converts categorical variables into numerical columns.

Some selected features may have zero missing values in this dataset, but the imputation step makes the workflow robust and keeps the same preprocessing rules inside cross-validation, model comparison, and final evaluation.

### Why use `ColumnTransformer`?

`ColumnTransformer` lets us apply different preprocessing steps to different columns.

In this notebook:

- numerical columns go through the numerical transformer
- categorical columns go through the categorical transformer

This keeps the workflow organized and helps prevent mistakes.


---

The preprocessing workflow looks like this:

Raw features  
→ split into numerical and categorical columns  
→ fill missing values  
→ scale numerical features  
→ one-hot encode categorical features  
→ send processed features into a machine learning model

---


In [ ]:
# OneHotEncoder changed the parameter name from "sparse" to "sparse_output"
# in newer scikit-learn versions.
# This helper function keeps the notebook compatible across versions.
def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

# Preprocessing steps for numerical features:
# 1. Fill missing values with the median
# 2. Standardize features so they are on a similar scale
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

# Preprocessing steps for categorical features:
# 1. Fill missing values with the most frequent category
# 2. Convert categories into one-hot encoded numerical columns
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_one_hot_encoder()),
])

# Apply the correct preprocessing pipeline to each group of columns
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## 11. Helper function for model evaluation

We will train and compare several classification models in this notebook.

Instead of writing the same evaluation code again and again, we define a helper function called `evaluate_binary_classifier()`.

This function takes three main inputs:

- a trained model
- the test features, `X_test`
- the test labels, `y_test`

Then it computes and prints several common evaluation metrics for binary classification.

> **What does this function calculate?**
>
> It reports **accuracy, precision, recall, and F1 score**, and prints a `classification_report` with precision, recall, and F1 broken out for both classes (`Slow adoption` and `Fast adoption`).
>
> The figure below defines each metric and shows how it is computed from the confusion matrix. The key contrast: **precision** asks "when the model predicts fast adoption, how often is it right?" while **recall** asks "of all truly fast-adoption pets, how many did the model find?"

<div align="center">
  <img src="images/precision_recall.png" width="500" alt="Confusion matrix and classification metrics: TN, FP, FN, TP, and the formulas for accuracy, precision, recall, and F1 score">
</div>

In [ ]:
def evaluate_binary_classifier(model, X_test, y_test, model_name="Model"):
    """Print core classification metrics and return them as a dictionary."""
    pred = model.predict(X_test)

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    }

    print(f"===== {model_name} =====")
    for k, v in metrics.items():
        if k != "Model":
            print(f"{k:10s}: {v:.3f}")

    print()
    print("Classification report:")
    print(classification_report(
        y_test,
        pred,
        target_names=["Slow adoption", "Fast adoption"],
        zero_division=0,
    ))

    return metrics


## 12. Train a first classifier: Logistic Regression

Logistic Regression is often a good first model for classification problems because it is:

- fast to train
- widely used
- relatively interpretable
- a strong baseline for many tabular datasets

Here, we combine two steps into one `Pipeline`:

1. `preprocess`: applies the preprocessing steps we created earlier
2. `model`: trains the Logistic Regression classifier

This means the raw input features will first be cleaned, scaled, and one-hot encoded, and then passed into the classifier.



---

> Why use `class_weight="balanced"`?

Our target variable is imbalanced: slow-adoption pets are more common than fast-adoption pets.

If we train the model without adjustment, it may focus too much on the majority class.

The argument `class_weight="balanced"` tells Logistic Regression to give more weight to the minority class during training. In this case, errors on fast-adoption pets receive more attention than they would in an unweighted model.


In [ ]:
log_reg_clf = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced")),
])

log_reg_clf.fit(X_train, y_train)

log_reg_metrics = evaluate_binary_classifier(
    log_reg_clf,
    X_test,
    y_test,
    model_name="Logistic Regression",
)

---
The Logistic Regression model gives the following overall performance:

- **Accuracy = 0.614**  
  The model correctly predicts about 61.4% of all test examples.

- **Precision = 0.327** for fast adoption  
  Among pets predicted as fast adoption, about 32.7% were actually fast adoption.

- **Recall = 0.620** for fast adoption  
  Among all truly fast-adoption pets, the model correctly identified about 62.0%.

- **F1 score = 0.428**  
  This combines precision and recall into one score.

The model finds many fast-adoption pets, but it also produces many false positives. In other words, it tends to label many pets as fast adoption, even when they are actually slow adoption.

---


The classification report shows precision, recall, and F1 score separately for each class.

For `Slow adoption`:

- Precision is **0.84**
- Recall is **0.61**
- F1 score is **0.71**

For `Fast adoption`:

- Precision is **0.33**
- Recall is **0.62**
- F1 score is **0.43**

This tells us that the model performs better on the slow-adoption class than on the fast-adoption class. This is common when one class is more frequent than the other.

The `support` column shows how many examples of each class are in the test set:

- 2,299 slow-adoption pets
- 700 fast-adoption pets

---


<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

1. Is 61.4% accuracy good enough for this task?
2. Why is the precision for fast adoption relatively low?
3. If a shelter wants to identify pets likely to be adopted quickly, would recall or precision matter more?
4. What might happen if we remove `class_weight="balanced"`?

</details>

In [ ]:
log_reg_unweighted = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", LogisticRegression(max_iter=1000))
])

log_reg_unweighted.fit(X_train, y_train)

evaluate_binary_classifier(
    log_reg_unweighted,
    X_test,
    y_test,
    model_name="Logistic Regression without class_weight"
)

## 13. Confusion matrix

Accuracy gives us one overall number, but it does not tell us what kinds of mistakes the model makes.

A **confusion matrix** shows the relationship between the true labels and the predicted labels.

In this plot:

- Rows are the true labels.
- Columns are the predicted labels.
- Each cell shows the number of examples in that category.

This helps us see whether the model is confusing slow-adoption pets with fast-adoption pets, or the other way around.

In [ ]:
y_pred = log_reg_clf.predict(X_test)

ConfusionMatrixDisplay.from_predictions(
    y_test,
    y_pred,
    display_labels=["Slow adoption", "Fast adoption"],
)
plt.title("Confusion Matrix: Logistic Regression")
plt.show()

This confusion matrix shows that the model correctly identifies **434** fast-adoption pets, but misses **266** fast-adoption pets.

It also incorrectly predicts **892** slow-adoption pets as fast adoption.

The model predicts `Fast adoption` often, which increases recall but also creates more false positives.

That helps increase recall for fast adoption, because the model catches more truly fast-adopted pets. However, it also lowers precision, because many predicted fast-adoption pets are actually slow adoption.

<details>
<summary><b>Discussion questions</b> (optional — click to expand)</summary>

1. How many fast-adoption pets did the model correctly identify?
2. How many fast-adoption pets did the model miss?
3. How many slow-adoption pets were incorrectly predicted as fast adoption?
4. If a shelter wants to identify as many truly fast-adoption pets as possible, which metric matters most?
5. If a shelter wants its fast-adoption predictions to be highly reliable, which metric matters most?

</details>